# Workload exploration and jumps

Cleaned from your explore notebook. Reading-based findings use the first 110 observations. Source lifecycle metadata is descriptive only.

In [ ]:
from pathlib import Path
import os
import sys
import numpy as np
import pandas as pd
from IPython.display import display, Image

def is_project(path):
    return (path / 'main.py').is_file() and (path / 'src/workload/data.py').is_file()

anchor = Path(os.environ.get('VM_PROJECT_ROOT', str(Path.cwd()))).expanduser().resolve()
ROOT = next((p for p in [anchor, *anchor.parents] if is_project(p)), None)
if ROOT is None:
    matches = [p for p in anchor.iterdir() if p.is_dir() and is_project(p)]
    if len(matches) != 1:
        raise RuntimeError('Open from the checkout/notebooks directory, or set VM_PROJECT_ROOT to the checkout path.')
    ROOT = matches[0]
sys.path.insert(0, str(ROOT / 'src'))
sys.path.insert(0, str(ROOT / 'notebooks'))
from workload.data import load_data, split, validate_predictions

# Set VM_DATA_DIR, or change this variable to the directory with the two raw files.
DATA_DIR = Path(os.environ.get('VM_DATA_DIR', str(ROOT / 'challenge_data'))).expanduser()
data = load_data(DATA_DIR)
n, future_times = split(data)
RESULTS = ROOT / 'src/results'
OUT = ROOT / 'notebooks/outputs'
OUT.mkdir(parents=True, exist_ok=True)
print(f'{n} training timestamps; {len(future_times)} forecast timestamps; raw files: {DATA_DIR}')


## Coverage, missingness and CPU distributions

**My observations and explanation:**

_Add your notes here._

In [ ]:
from workload.analysis import analyze
summary = analyze(data, n, OUT)
display(pd.Series(data.audit, name='Data audit').to_frame())
display(pd.Series(summary, name='Training findings').to_frame())
display(Image(filename=str(OUT / 'figures/cpu_distributions.png')))

## Jump analysis

Flag absolute five-minute changes above `median(abs(change)) + 2 * MAD(abs(change))`. This preserves your heuristic. It identifies unusual changes, not calibrated anomalies; adjacent flags can describe one event.

**My observations and explanation:**

_Add your notes here._

In [ ]:
from analysis_helpers import jump_analysis
jumps, attribution, jump_summary = jump_analysis(data, n, OUT)
display(pd.Series(jump_summary).to_frame('Value'))
display(jumps)
display(Image(filename=str(OUT / 'jumps.png')))

## Which groups contributed?

Group contributions are count-weighted percentage points; their changes sum to the fleet change. Stable observed membership makes these workload changes rather than changes in the represented VM count. Recurring intervals alone do not establish a stable season.

**My observations and explanation:**

_Add your notes here._

In [ ]:
largest = jump_summary['largest_absolute_change_timestamp']
display(attribution.loc[attribution.timestamp.eq(largest)].sort_values('contribution_change_pp', ascending=False))
display(jumps.minutes_since_previous_flag.value_counts().rename_axis('Minutes').to_frame('Intervals'))

## Lifetime and interpretation limits

Complete-trace selection produces no observed internal churn. Creation/deletion at source boundaries may be censored. The corrected active count includes idle VMs; `max_cpu > 0.01` measures a separate busy count, not the required active target.

**My observations and explanation:**

_Add your notes here._